# VGGT Point Map — Untargeted PGD Attack (paper-style L2 loss)

Companion to `depth_attack.ipynb`. Same threat model (L∞-bounded image perturbation, shared across views, 40 PGD steps); we target VGGT's **point map** head (`world_points`), which predicts a per-pixel 3D point in the first camera's world frame. Unlike the depth attack — which runs on a DTU scan — this notebook drives the attack on an **ETH3D** scene (high-resolution undistorted DSLR photos of an outdoor courtyard), so the point-map probe sees a different geometry distribution (large outdoor depths, wide baselines, sparse SfM ground truth).

Loss design — directly maximize the VGGT point-map training loss, with the **clean** precision $\Sigma_P^{\text{clean}}$ frozen as an importance weight. From the VGGT paper:

$$\mathcal{L}_{\text{pmap}} \;=\; \sum_{i=1}^{N} \bigl\| \Sigma_i^P \odot (\hat{P}_i - P_i) \bigr\| \;+\; \bigl\| \Sigma_i^P \odot (\nabla \hat{P}_i - \nabla P_i) \bigr\| \;-\; \alpha \log \Sigma_i^P$$

Our attack maximizes the first (regression) term, using the *clean* prediction $P_i = P_{\text{clean}}$ in place of the ground truth and a *detached* clean precision $\Sigma_i^P = \Sigma_{\text{clean}}$ as the per-pixel weight:

$$L_{\text{atk}} \;=\; \mathbb{E}_{i}\Bigl[\, \Sigma_i^{\text{clean}} \cdot \bigl\| P_{\text{adv}}^{(i)} - P_{\text{clean}}^{(i)} \bigr\|_2 \,\Bigr]$$

Why this is the right loss (and why the previous SI-3D divergence under-performed):
- **The attacker cannot hide behind dropped confidence.** Because $\Sigma$ is taken from the clean forward pass and detached, the only way to grow the loss is to actually displace `world_points`. The previous notebook's loss instead let the model lower $\Sigma_{\text{adv}}$ on the very pixels the attack damaged, so the standard MVSNet confidence filter just culled those points and the Chamfer barely moved.
- **The loss is unbounded above** (the per-pixel L2 residual has no ceiling — unlike the SI angular term which saturates at 2.0 per pixel). PGD therefore has a clear ascent direction at every step.
- **The per-pixel gradient $\partial L / \partial P_{\text{adv}}^{(i)}$ is exactly the unit vector $(P_{\text{adv}}^{(i)} - P_{\text{clean}}^{(i)})/\|P_{\text{adv}}^{(i)} - P_{\text{clean}}^{(i)}\|$ scaled by $\Sigma_i^{\text{clean}}$.** So PGD's sign-step is *pointed away from each clean point in 3D*, weighted by clean confidence.
- **Reflection/rotation/per-pixel displacement directions** all grow the loss. Even uniform inflation grows it — but ICP-with-scale at eval time absorbs uniform scale, so the *effective* metric is the per-pixel relative breakdown that PGD finds in practice.

Outputs go to `output/clean_pointmap/`, `output/adv_pointmap/`, and `images/adv_pointmap/` for `evaluate_attack.py --eth3d` to consume.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.utils as vutils

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 0
seed_everything(SEED)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)


In [ ]:
# ETH3D high-res training scene: dslr_images_undistorted/ holds 38 JPGs.
# We pick 8 of them — same view count as the DTU depth attack so per-view
# numbers stay comparable — and let load_and_preprocess_images downscale to
# 518-wide (≈518×350 here given the 6208×4134 originals).
ETH3D_SCENE = Path("/u/nleobandung/eth3d/courtyard")
IMAGE_DIR = ETH3D_SCENE / "images/dslr_images_undistorted"
NUM_IMAGES = 8

all_images = sorted(str(p) for p in IMAGE_DIR.iterdir() if p.suffix.upper() == ".JPG")
selected_images = sorted(random.sample(all_images, NUM_IMAGES))
print(f"Selected {len(selected_images)} images from {IMAGE_DIR}")
for p in selected_images:
    print(" ", p)

images = load_and_preprocess_images(selected_images).to(device).float()
images = images.unsqueeze(0)
B, S, C, H, W = images.shape
print("images:", images.shape, "range:", float(images.min()), float(images.max()))

In [ ]:
# Clean forward pass — the reference for the attack loss.
with torch.no_grad():
    clean = model(images)

P_clean  = clean["world_points"].float()        # [B, S, H, W, 3]
PC_clean = clean["world_points_conf"].float()   # [B, S, H, W]
print("world_points:", tuple(P_clean.shape),
      "norm mean:", float(P_clean.norm(dim=-1).mean()),
      " world_points_conf:", tuple(PC_clean.shape),
      "mean:", float(PC_clean.mean()))


In [ ]:
# Identical L_inf budget to the depth attack so the two probes are comparable.
EPS = 4.0 / 255.0
STEP = 1.0 / 255.0
STEPS = 40
RANDOM_INIT = True
SHARE_DELTA_ACROSS_VIEWS = True


In [ ]:
def project_delta(delta_tensor: torch.Tensor, images_tensor: torch.Tensor, eps: float) -> torch.Tensor:
    """Project delta onto the L_inf ball of radius eps, intersected with the
    box that keeps `images + delta` in [0, 1] across every view (delta is
    broadcast over the S axis when shared).
    """
    shared = delta_tensor.shape[1] == 1 and images_tensor.shape[1] > 1
    if shared:
        lo_pix = (-images_tensor).amax(dim=1, keepdim=True)
        hi_pix = (1.0 - images_tensor).amin(dim=1, keepdim=True)
    else:
        lo_pix = -images_tensor
        hi_pix = 1.0 - images_tensor
    lo = torch.maximum(lo_pix, torch.full_like(lo_pix, -eps))
    hi = torch.minimum(hi_pix, torch.full_like(hi_pix, eps))
    return torch.clamp(delta_tensor, lo, hi)


In [ ]:
def point_attack_loss(P_adv: torch.Tensor,
                      P_clean: torch.Tensor,
                      conf_clean: torch.Tensor) -> torch.Tensor:
    """Maximize the VGGT point-map training regression term, using clean
    confidence as a frozen per-pixel weight.

        L = mean_i ( Σ_i^clean · || P_adv_i - P_clean_i ||_2 )

    Why this exact form:
      * It is exactly the L_pmap regression term from the paper, with the
        ground truth replaced by the clean prediction and α log Σ dropped
        (that regularizer disciplines the model's parameters during
        training; it has nothing to grade an input attack on).
      * conf_clean is .detach()'d. The previous SI-3D loss let the attacker
        win by lowering its own predicted confidence on the pixels it
        damaged — those pixels then got culled by the standard MVSNet
        confidence filter and the Chamfer barely moved. Freezing Σ at
        clean values closes that escape hatch: the attacker has to move
        the 3D coordinates themselves.
      * Per-pixel gradient ∂L/∂P_adv_i = Σ_i^clean · (P_adv_i - P_clean_i)
                                          / ||P_adv_i - P_clean_i||_2,
        so the sign-step PGD update points each predicted point *away
        from its clean value in 3D*, weighted by the clean precision.
      * Unbounded above — PGD has a clear ascent direction at every step.
        (The earlier SI angular term saturated at 2.0/pixel.)
    """
    w = conf_clean.detach()                           # [B, S, H, W]
    diff = (P_adv - P_clean).norm(dim=-1)             # [B, S, H, W], L2 per pixel
    return (w * diff).mean()


In [ ]:
delta_shape = (B, 1, C, H, W) if SHARE_DELTA_ACROSS_VIEWS else (B, S, C, H, W)
delta = torch.zeros(delta_shape, device=device, dtype=torch.float32)
if RANDOM_INIT:
    delta.data.uniform_(-EPS, EPS)
delta.data.copy_(project_delta(delta.data, images, EPS))
delta.requires_grad_(True)


In [ ]:
torch.cuda.empty_cache()
loss_history = []

for i in range(STEPS):
    if delta.grad is not None:
        delta.grad.detach_()
        delta.grad.zero_()

    x_adv = images + delta
    preds = model(x_adv)
    P_adv = preds["world_points"].float()

    loss = point_attack_loss(P_adv, P_clean, PC_clean)

    (g,) = torch.autograd.grad(loss, delta)

    with torch.no_grad():
        delta.add_(STEP * g.sign())
        delta.copy_(project_delta(delta, images, EPS))

    loss_history.append(float(loss.detach()))
    if i % 5 == 0 or i == STEPS - 1:
        with torch.no_grad():
            linf = float(delta.abs().max())
        print(f"step {i:3d}  loss={loss_history[-1]:.6f}  ||delta||_inf={linf:.5f}")

adv_images_t = (images + delta).detach()
assert float(adv_images_t.min()) >= -1e-6
assert float(adv_images_t.max()) <= 1.0 + 1e-6
assert float((adv_images_t - images).abs().max()) <= EPS + 1e-6


In [ ]:
with torch.no_grad():
    advp = model(adv_images_t)

P_adv_t  = advp["world_points"].float()
PC_adv_t = advp["world_points_conf"].float()

# Unaligned per-pixel residual diagnostics — the alignment-based Acc/Comp
# Chamfer metric is the evaluator's job. These help spot which views the
# attack hit hardest.
n_adv   = P_adv_t.norm(dim=-1).clamp(min=1e-6)
n_clean = P_clean.norm(dim=-1).clamp(min=1e-6)
diff_l2 = (P_adv_t - P_clean).norm(dim=-1)                      # [B,S,H,W]
rel_err = diff_l2 / n_clean                                     # scale-invariant per pixel
log_r   = torch.log(n_adv) - torch.log(n_clean)
cos_sim = (P_adv_t * P_clean).sum(dim=-1) / (n_adv * n_clean)
cos_sim = cos_sim.clamp(-1.0 + 1e-6, 1.0 - 1e-6)

w = PC_clean.detach()
per_view_loss_pixelmean = diff_l2.mean(dim=(-1, -2))            # [B,S], unweighted
per_view_loss_confw     = (w * diff_l2).mean(dim=(-1, -2)) / w.mean(dim=(-1, -2)).clamp(min=1e-6)

summary = {
    "mean_norm_clean":              float(n_clean.mean()),
    "mean_norm_adv":                float(n_adv.mean()),
    "mean_abs_point_err":           float(diff_l2.mean()),          # ||P_adv - P_clean||_2 averaged over all pixels
    "mean_rel_point_err":           float(rel_err.mean()),          # divided by ||P_clean||
    "mean_abs_log_norm_err":        float(log_r.abs().mean()),
    "mean_angular_err":             float((1.0 - cos_sim).mean()),  # 1 - cos θ
    "mean_cos_sim_clean_adv":       float(cos_sim.mean()),
    "attack_loss_final":            float((w * diff_l2).mean()),    # the actual quantity PGD maximized
    "mean_conf_clean":              float(PC_clean.mean()),
    "mean_conf_adv":                float(PC_adv_t.mean()),
    "linf_delta":                   float(delta.detach().abs().max()),
    "l2_delta_per_pixel":           float(delta.detach().pow(2).mean().sqrt()),
    "per_view_l2_residual":         per_view_loss_pixelmean.squeeze(0).tolist(),
    "per_view_l2_residual_confw":   per_view_loss_confw.squeeze(0).tolist(),
}
print(json.dumps(summary, indent=2))


In [ ]:
out_dir = Path("output")
out_dir.mkdir(exist_ok=True)
out_clean_dir = out_dir / "clean_pointmap";  out_clean_dir.mkdir(exist_ok=True)
out_adv_dir   = out_dir / "adv_pointmap";    out_adv_dir.mkdir(exist_ok=True)
img_dir       = Path("images/adv_pointmap"); img_dir.mkdir(parents=True, exist_ok=True)

def _np(t):
    return t.detach().cpu().numpy()

def _pose_enc_list(pred):
    return np.stack([_np(p) for p in pred["pose_enc_list"]], axis=0)

image_size_hw = (H, W)
clean_extr, clean_intr = pose_encoding_to_extri_intri(clean["pose_enc"].float(), image_size_hw)
adv_extr,   adv_intr   = pose_encoding_to_extri_intri(advp["pose_enc"].float(),  image_size_hw)

# We record the relative paths that appear in ETH3D's images.txt
# ("dslr_images_undistorted/<basename>.JPG") so the evaluator can look up the
# IMAGE_IDs and filter points3D.txt to GT points visible in our N selected
# views.  Without this filter, completeness would be biased by all the GT
# points outside our camera frustums.
image_rel_paths = [
    f"dslr_images_undistorted/{Path(p).name}" for p in selected_images
]

# Same field layout as the depth attack so evaluate_attack.py loads both with
# no special casing. depth/depth_conf are kept (from the same forward passes)
# so a downstream user can also check the collateral damage on the depth head.
np.savez_compressed(
    out_clean_dir / "clean_pointmap.npz",
    world_pts=_np(P_clean),
    world_pts_conf=_np(PC_clean),
    depth=_np(clean["depth"]),
    depth_conf=_np(clean["depth_conf"]),
    pose_enc=_np(clean["pose_enc"]),
    pose_enc_list=_pose_enc_list(clean),
    extrinsics=_np(clean_extr),
    intrinsics=_np(clean_intr),
    image_rel_paths=np.array(image_rel_paths),
    scene_dir=np.array(str(ETH3D_SCENE)),
)

np.savez_compressed(
    out_adv_dir / "adv_pointmap.npz",
    world_pts=_np(P_adv_t),
    world_pts_conf=_np(PC_adv_t),
    depth=_np(advp["depth"]),
    depth_conf=_np(advp["depth_conf"]),
    pose_enc=_np(advp["pose_enc"]),
    pose_enc_list=_pose_enc_list(advp),
    extrinsics=_np(adv_extr),
    intrinsics=_np(adv_intr),
    delta=_np(delta),
    adv_images=_np(adv_images_t),
    clean_images=_np(images),
    loss_history=np.array(loss_history, dtype=np.float32),
    per_view_l2_residual=_np(per_view_loss_pixelmean),
    per_view_l2_residual_confw=_np(per_view_loss_confw),
    per_pixel_l2_residual=_np(diff_l2),
    image_rel_paths=np.array(image_rel_paths),
    scene_dir=np.array(str(ETH3D_SCENE)),
)

adv_S = adv_images_t.squeeze(0)
for s in range(adv_S.shape[0]):
    vutils.save_image(adv_S[s], img_dir / f"adv_s{s}.png", normalize=False)

env = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

metadata = {
    "seed": SEED,
    "model": "facebook/VGGT-1B",
    "scene": str(ETH3D_SCENE),
    "selected_images": selected_images,
    "image_rel_paths": image_rel_paths,
    "image_shape": list(images.shape),
    "attack": {
        "target": "world_points (point-map head)",
        "type": "PGD (sign step, L_inf ball)",
        "loss": (
            "L = mean( Sigma_clean * ||P_adv - P_clean||_2 ).  "
            "Paper's L_pmap regression term, with the clean prediction in place "
            "of the ground truth and clean predicted precision used as a "
            "DETACHED per-pixel weight so the attacker cannot win by "
            "lowering predicted confidence."
        ),
        "eps": EPS,
        "step": STEP,
        "steps": STEPS,
        "random_init": RANDOM_INIT,
        "share_delta_across_views": SHARE_DELTA_ACROSS_VIEWS,
    },
    "summary": summary,
    "env": env,
}
with open(out_dir / "pointmap_attack_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("wrote:")
print(" ", out_clean_dir / "clean_pointmap.npz")
print(" ", out_adv_dir / "adv_pointmap.npz")
print(" ", out_dir / "pointmap_attack_metadata.json")
print(" ", img_dir, "(PNG per view)")